#Initialisations

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, add_months, substring
from pyspark.sql.types import StringType

In [0]:
RENAME_MAP = {
    "prd_id" : "product_id",
    "prd_key" : "product_key",
    "prd_nm" : "product_name",
    "prd_cost" : "product_cost",
    "prd_line" : "product_line",
    "prd_start_dt" : "product_start_date",
    "prd_end_dt" : "product_end_date"
}

#Reading from the Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info_raw")

#Transformations

##1. TRIM Text column to remove empty spaces

In [0]:
for field in df.schema:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

##2. Normalising prd_line to something more descriptive and freindly

In [0]:
df = (
    df.withColumn(
        "prd_line",
        F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
        .when(F.upper(F.col("prd_line")) == "R", "Road")
        .when(F.upper(F.col("prd_line")) == "T", "Touring")
        .when(F.upper(F.col("prd_line")) == "S", "Sport")
        .otherwise(F.col("prd_line"))
    )
)

##3. prd_end_dt is 6 years back, we will add the years to the date to fix the issue

In [0]:
df = (
    df.withColumn(
        "prd_end_dt",
        F.when(F.col("prd_end_dt").isNotNull(), add_months(F.col("prd_end_dt"), 48))
        .otherwise(F.col("prd_end_dt"))
    )
)

##4. Standadising business key id to allowing joining table

In [0]:
#This is for connecting with the other product table from erp
df = df.withColumn("category_id", substring(F.col("prd_key"),1,5))

#This will be for joining the table with Fact sales table
df = df.withColumn("sales_product_key", substring(F.col("prd_key"),7,10))

##5. Renaming the column headers to something more friendly

In [0]:
for old_name,new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name,new_name)

#Writting to the silver layer

In [0]:
(
  df.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")	
  .saveAsTable("silver.crm_products")
)